## Mujoco Jacobian
- Get Jacobian of the specific body
- Get pseudo-inverse of Jacobian
    - Damped Least Squares
    - Singular Value Decomposition

### Singular Value Decomposition
- Decompose jacobian into S, V and sigma
- V: direction of motion in joint space, U: corresponding motion direction in end-effector space
- Sigma: Contains singular values (degree of amplification)
    - large singular value -> small joint move makes large 
    - Thresholding sigma to avoid singular value
- $J = U \Sigma V^T$
- $J^+ = V \Sigma^+ U^T$
</br>

### Damped Least Squares
- pseudo-inverse of Jacobian
- Damping factor: reduce singularity issue
    - lambda damping
    - e with error vector
- $\Delta\theta = J^T(JJ^T + \lambda^2I)^{-1} \vec{e}$

Create UR environment

In [1]:
import mujoco
import mujoco_viewer # new viewer
import numpy as np
import time

In [ ]:
model_path = "../assets/ur5e_mjcf/scene.xml"

# declare model & data
model = mujoco.MjModel.from_xml_path(model_path)
data = mujoco.MjData(model)

In [ ]:
def get_body_names (model, data):
    body_names = [mujoco.mj_id2name(model,mujoco.mjtObj.mjOBJ_BODY, body_idx) for body_idx in range(model.nbody)]
    return body_names

body_names = get_body_names(model, data)

print(body_names)

['world', 'base', 'shoulder_link', 'upper_arm_link', 'forearm_link', 'wrist_1_link', 'wrist_2_link', 'wrist_3_link']


Get Mujoco Jacobian

In [ ]:
""" GET MUJOCO JACOBIAN """

body_name = "wrist_3_link"

# initialize positional & rotational jacobian
Jacobian_p = np.zeros((3,model.nu))
Jacobian_r = np.zeros((3,model.nu))
# get jacobian of end-effector

mujoco.mj_resetData(model, data)
mujoco.mj_forward(model, data)

mujoco.mj_jacBody(model, data, Jacobian_p, Jacobian_r, data.body(body_name).id)
print(Jacobian_p) 

# shape: 3x6 because 6 dof arm

"""
IMPORTANT: jacobian is not calculated if no step / forward 
"""
# mujoco.mj_jac(model, data, p, r, )

[[-8.1700000e-01 -4.4408921e-17 -4.4408921e-17 -4.4408921e-17
   0.0000000e+00  0.0000000e+00]
 [-1.3400000e-01 -1.0000000e-01 -1.0000000e-01 -1.0000000e-01
   0.0000000e+00  0.0000000e+00]
 [ 0.0000000e+00 -8.1700000e-01 -3.9200000e-01  0.0000000e+00
   0.0000000e+00  0.0000000e+00]]


### Calculate delta theta with inverse jacobian

In [13]:
U, S, V = np.linalg.svd(Jacobian_p, compute_uv=True)

In [17]:
print(f"U, S, V from SVD:\nU: {U}\nS (vector): {S}\nV: {V}")

U, S, V from SVD:
U: [[-0.10311419  0.98124184  0.1628862 ]
 [-0.16373586  0.14478032 -0.97582233]
 [-0.98110042 -0.12729145  0.14573557]]
S (vector): [0.91724043 0.82682662 0.10782269]
V: [[ 1.15765614e-01  8.91731985e-01  4.37142692e-01  1.78509208e-02
   0.00000000e+00  0.00000000e+00]
 [-9.93043913e-01  1.08268259e-01  4.28387451e-02 -1.75103597e-02
   0.00000000e+00  0.00000000e+00]
 [-2.14967301e-02 -1.99250545e-01  3.75189016e-01  9.05025035e-01
   0.00000000e+00  0.00000000e+00]
 [-9.24446373e-33  3.91651976e-01 -8.16274654e-01  4.24622678e-01
   0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   1.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  1.00000000e+00]]
